# 05 — Feedback loop cải tiến (OOD error analysis)

Giai đoạn 5 (`docs/PLAN.md`) — spec: `docs/specs/g5-feedback-loop.md`.

Giai đoạn 4 không tìm ra cặp lớp nào bị nhầm trên test set gốc (0/101 sai)
nên hướng cải thiện không nhắm vào "lỗi ở mục 4" theo nghĩa đen — thay vào
đó, notebook này đo lỗi thật của model baseline (`yolov8n_v1_baseline`,
augmentation ON) trên ảnh **ngoài phân bố dataset Roboflow** (OOD), rồi tuỳ
kết quả mà retrain với 1 tweak augmentation cụ thể hoặc kết luận model đã
đủ robust.

**Chạy trên Colab** (cần GPU nếu retrain ở phần dưới — Runtime → Change
runtime type → T4 GPU).


## Setup — mount Drive + cd vào repo

Giống hệt các notebook trước.


In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
BEST_WEIGHTS = "runs/detect/yolov8n_v1_baseline/weights/best.pt"
assert os.path.isfile(BEST_WEIGHTS), (
    f"{BEST_WEIGHTS} chưa có — chạy notebooks/02_train_detector.ipynb "
    "(Giai đoạn 2/3) trước để có checkpoint baseline."
)
OOD_DIR = "data/ood_samples"
assert os.path.isdir(OOD_DIR) and any(os.scandir(OOD_DIR)), (
    f"{OOD_DIR}/ chưa có ảnh — làm T5.2a trước (xem "
    "docs/specs/g5-feedback-loop.md): chụp/tải ảnh yoga ngoài dataset "
    "Roboflow, bỏ vào đây rồi chạy lại."
)


## Đồng bộ code mới nhất


In [ ]:
assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git pull


## Cài dependencies


In [ ]:
!pip install -q -r requirements.txt
import torch
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## T5.2a — Tập ảnh OOD

Ảnh đã bỏ vào `data/ood_samples/` theo 1 trong 2 kiểu (xem spec T5.2a):
- File rời, tên bắt đầu bằng tên lớp: `tree_01.jpg`
- Hoặc thư mục con theo lớp: `data/ood_samples/<ten_lop_goc>/*.jpg`

`CLASS_NAME_MAP` map tên lớp gốc (file/thư mục) sang đúng tên lớp model.
Tên đã khớp sẵn (`bridge`, `downward`, `plank`, `shoulderstand`, `tree`)
không cần khai lại — chỉ cần khai tên lệch, vd dataset Kaggle
`niharika41298/yoga-poses-dataset` dùng `downdog` thay vì `downward`.
**Sửa dict này cho khớp đúng dataset bạn dùng trước khi chạy cell dưới.**


In [ ]:
CLASS_NAME_MAP = {
    "downdog": "downward",
    # thêm mapping khác nếu dataset bạn dùng đặt tên lớp khác, vd:
    # "shoulder_stand": "shoulderstand",
}
MODEL_CLASSES = {"bridge", "downward", "plank", "shoulderstand", "tree"}


## T5.2b — Đo baseline OOD accuracy

`collect_ood_samples()` đọc cả 2 kiểu tổ chức thư mục ở trên, suy nhãn thật
từ tên file (`<lop>_...`) hoặc tên thư mục cha, map qua `CLASS_NAME_MAP`,
bỏ qua (kèm cảnh báo) ảnh không suy ra được lớp nào khớp `MODEL_CLASSES`.


In [ ]:
from pathlib import Path

OOD_ROOT = Path(OOD_DIR)
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def infer_true_class(image_path: Path) -> str | None:
    """Suy nhãn thật từ tên thư mục cha (nếu khác OOD_ROOT) hoặc tiền tố tên file."""
    candidates = []
    if image_path.parent != OOD_ROOT:
        candidates.append(image_path.parent.name)
    candidates.append(image_path.stem.split("_")[0])

    for raw in candidates:
        mapped = CLASS_NAME_MAP.get(raw, raw)
        if mapped in MODEL_CLASSES:
            return mapped
    return None


def collect_ood_samples() -> list[dict]:
    samples = []
    skipped = []
    for image_path in sorted(OOD_ROOT.rglob("*")):
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        true_cls = infer_true_class(image_path)
        if true_cls is None:
            skipped.append(image_path)
            continue
        samples.append({"path": image_path, "true": true_cls})

    if skipped:
        print(f"CẢNH BÁO: bỏ qua {len(skipped)} ảnh không suy ra được nhãn hợp lệ:")
        for p in skipped:
            print(f"  {p} — kiểm tra CLASS_NAME_MAP hoặc tên file/thư mục")
    return samples


ood_samples = collect_ood_samples()
print(f"Đọc được {len(ood_samples)} ảnh OOD hợp lệ.")
assert len(ood_samples) >= 1, "Không có ảnh OOD hợp lệ nào — kiểm tra lại CLASS_NAME_MAP/tên file."


In [ ]:
from ultralytics import YOLO

yolo = YOLO(BEST_WEIGHTS)


def top_class_from_result(result) -> tuple[str | None, float]:
    if len(result.boxes) == 0:
        return None, 0.0
    top_idx = int(result.boxes.conf.argmax())
    class_id = int(result.boxes.cls[top_idx])
    conf = float(result.boxes.conf[top_idx])
    return result.names[class_id], conf


def evaluate_ood(samples: list[dict]) -> list[dict]:
    """Chạy predict() đúng 1 lần/ảnh, giữ lại `result` để cell vẽ lưới bên dưới
    dùng lại (result.plot()) thay vì predict lại lần 2 tốn thời gian."""
    evaluated = []
    for s in samples:
        result = yolo.predict(source=str(s["path"]), verbose=False)[0]
        pred_cls, conf = top_class_from_result(result)
        evaluated.append(
            {**s, "pred": pred_cls, "conf": conf, "correct": pred_cls == s["true"], "result": result}
        )
    return evaluated


def print_ood_report(evaluated: list[dict], label: str) -> float:
    n_correct = sum(e["correct"] for e in evaluated)
    accuracy = n_correct / len(evaluated) if evaluated else 0.0
    print(f"--- OOD accuracy {label}: {n_correct}/{len(evaluated)} = {accuracy:.1%} ---")
    for e in evaluated:
        status = "ĐÚNG" if e["correct"] else "SAI "
        print(f"  {status}  {e['path'].name:30s} thật={e['true']:15s} đoán={str(e['pred']):15s} conf={e['conf']:.2f}")
    return accuracy


ood_results_before = evaluate_ood(ood_samples)
ood_accuracy_before = print_ood_report(ood_results_before, "TRƯỚC (baseline)")


Lưới ảnh có box vẽ đè, để soi bằng mắt (không chỉ nhìn số) — hữu ích để xác
định *loại* lỗi (ảnh tối, người nhỏ, nền lộn xộn, mờ...) cho T5.3.


In [ ]:
import math

import matplotlib.pyplot as plt

n = len(ood_results_before)
n_cols = min(4, n)
n_rows = math.ceil(n / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten() if n > 1 else [axes]
for ax, r in zip(axes, ood_results_before):
    ax.imshow(r["result"].plot()[:, :, ::-1])  # BGR -> RGB, dùng lại result đã có từ evaluate_ood()
    ax.axis("off")
    status = "ĐÚNG" if r["correct"] else "SAI"
    ax.set_title(f"{status}: thật={r['true']}, đoán={r['pred']} ({r['conf']:.2f})", fontsize=9)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## T5.3 — Áp tweak augmentation (nếu có lỗi rõ rệt)

Nhìn bảng + lưới ảnh ở trên, xác định `OBSERVED_ISSUE` khớp đúng 1 dòng
trong bảng mapping (xem `docs/specs/g5-feedback-loop.md`). Nếu OOD accuracy
đã cao (vd ≥ 90%) và không thấy lỗi rõ rệt, để `OBSERVED_ISSUE = None` —
cell dưới sẽ tự bỏ qua retrain và kết luận baseline đã đủ robust.

**Sửa `OBSERVED_ISSUE` cho khớp quan sát thật của bạn rồi mới chạy tiếp.**


In [ ]:
# Bảng mapping cố định — xem lý do từng dòng trong docs/specs/g5-feedback-loop.md.
# Chọn đúng 1 key khớp lỗi thấy NHIỀU NHẤT trên ảnh OOD, hoặc None nếu robust.
AUGMENTATION_TWEAKS = {
    "dark_lighting": {"hsv_v": 0.8},       # mặc định Ultralytics: 0.4
    "small_person": {"scale": 0.9},        # mặc định Ultralytics: 0.5
    "cluttered_background": {"mosaic": 1.0, "copy_paste": 0.3},  # mosaic mặc định đã là 1.0 -> chủ yếu bật copy_paste
    "blur_or_odd_angle": {"degrees": 20.0, "shear": 5.0},  # mặc định: degrees=10 (T1.5), shear=0.0
}

OBSERVED_ISSUE = None  # <-- đổi thành 1 key ở trên (vd "dark_lighting") nếu thấy lỗi rõ rệt

assert OBSERVED_ISSUE is None or OBSERVED_ISSUE in AUGMENTATION_TWEAKS, (
    f"OBSERVED_ISSUE={OBSERVED_ISSUE!r} không khớp key nào trong AUGMENTATION_TWEAKS."
)


In [ ]:
from src.models.train import train

if OBSERVED_ISSUE is None:
    print(
        "Không có lỗi rõ rệt trên tập OOD — bỏ qua retrain. Kết luận: baseline "
        f"(yolov8n_v1_baseline) đã đủ robust ở mức OOD accuracy {ood_accuracy_before:.1%} "
        "đã đo ở trên."
    )
    results_feedback = None
else:
    tweak_kwargs = AUGMENTATION_TWEAKS[OBSERVED_ISSUE]
    print(f"Áp tweak cho lỗi '{OBSERVED_ISSUE}': {tweak_kwargs}")
    results_feedback = train(
        data="data/raw/yoga_v1/data.yaml",
        model="yolov8n.pt",
        seed=42,
        epochs=50,
        name="yolov8n_v1_feedback_v2",
        exist_ok=True,
        **tweak_kwargs,
    )
    print("save_dir:", results_feedback.save_dir)


## T5.4 — Bảng so sánh trước/sau

Chỉ có ý nghĩa nếu đã retrain ở T5.3 (`OBSERVED_ISSUE is not None`) — nếu
không, cell dưới tự bỏ qua.

`BASELINE_MAP50`/`BASELINE_MAP50_95` là số thật đã ghi nhận ở Giai đoạn 3
(`docs/problem_statement.md`, config ON) — dùng lại thay vì validate lại
`yolov8n_v1_baseline` cho tốn thời gian Colab.


In [ ]:
BASELINE_MAP50 = 0.9922
BASELINE_MAP50_95 = 0.8352

if results_feedback is None:
    print("Không retrain (xem T5.3) — không có bảng trước/sau.")
else:
    yolo = YOLO("runs/detect/yolov8n_v1_feedback_v2/weights/best.pt")
    ood_results_after = evaluate_ood(ood_samples)
    ood_accuracy_after = print_ood_report(ood_results_after, "SAU (feedback_v2)")

    val_after = yolo.val(data="data/raw/yoga_v1/data.yaml")

    import pandas as pd

    comparison = pd.DataFrame(
        [
            {
                "run": "yolov8n_v1_baseline (trước)",
                "OOD accuracy": f"{ood_accuracy_before:.1%}",
                "mAP@0.5": BASELINE_MAP50,
                "mAP@0.5:0.95": BASELINE_MAP50_95,
            },
            {
                "run": "yolov8n_v1_feedback_v2 (sau)",
                "OOD accuracy": f"{ood_accuracy_after:.1%}",
                "mAP@0.5": val_after.box.map50,
                "mAP@0.5:0.95": val_after.box.map,
            },
        ]
    ).set_index("run")
    comparison


## T5.5 — Kết luận

*(Điền tay sau khi có số liệu thật ở trên — xem hướng dẫn trong
`docs/specs/g5-feedback-loop.md` mục "Cách bạn tự test".)*
